In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:99% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render ul li{font-size:22pt; line-height:30px;}
div.output {font-size:22pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:22pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

In [9]:
# ===============================
# 📌 기본 라이브러리
# ===============================
import pandas as pd
import numpy as np

# ===============================
# 📌 시각화 라이브러리
# ===============================
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 📌 경고 메시지 제거
# ===============================
import warnings
warnings.filterwarnings("ignore")

# ===============================
# 📌 한글 폰트 설정 (Windows)
# ===============================
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


In [10]:
# ===============================
# 📌 CSV에서 불러올 컬럼 지정
# (불필요한 컬럼 제거 → 메모리 절약)
# ===============================
use_cols = [
    '인허가일자',
    '상세영업상태명',
    '폐업일자',
    '소재지전체주소',
    '도로명전체주소',
    '업태구분명',
    '좌표정보x(epsg5174)',
    '좌표정보y(epsg5174)'
]


In [11]:
# ===============================
# 📌 데이터 로드
# ===============================
df = pd.read_csv(
    r"C:\Users\Admin\Desktop\미니프로젝트파일\6110000_서울특별시_07_24_04_P_일반음식점.csv",
    encoding="cp949",
    usecols=use_cols,
    low_memory=False
)

In [12]:
df.head()


,인허가일자,상세영업상태명,폐업일자,소재지전체주소,도로명전체주소,업태구분명,좌표정보x(epsg5174),좌표정보y(epsg5174)
0,2024-05-30,영업,NaN,서울특별시 강북구 수유동 229-46,"서울특별시 강북구 도봉로87길 11, 지하1층 (수유동)",기타,202125.400308,459553.074675
1,2025-05-02,영업,NaN,서울특별시 종로구 관훈동 155-2,"서울특별시 종로구 인사동길 49, 3층 4호 (관훈동)","외국음식전문점(인도,태국등)",198498.918633,452473.403750
2,2025-05-02,영업,NaN,서울특별시 종로구 관훈동 155-2,"서울특별시 종로구 인사동길 49, 4층 401호 (관훈동)",한식,198498.918633,452473.403750
3,2025-05-02,영업,NaN,서울특별시 종로구 인사동 241,"서울특별시 종로구 종로11길 9-10, 2층 (인사동)",기타,198625.513334,452075.105119
4,2024-05-31,영업,NaN,서울특별시 양천구 신월동 60-33,"서울특별시 양천구 화곡로 96, 1층 (신월동)",한식,185121.904558,448581.231912


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527872 entries, 0 to 527871
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   인허가일자            527872 non-null  object 
 1   상세영업상태명          527872 non-null  object 
 2   폐업일자             406206 non-null  object 
 3   소재지전체주소          527644 non-null  object 
 4   도로명전체주소          277927 non-null  object 
 5   업태구분명            527854 non-null  object 
 6   좌표정보x(epsg5174)  495424 non-null  float64
 7   좌표정보y(epsg5174)  495424 non-null  float64
dtypes: float64(2), object(6)
memory usage: 32.2+ MB


In [14]:
df.isna().sum()


인허가일자                   0
상세영업상태명                 0
폐업일자               121666
소재지전체주소               228
도로명전체주소            249945
업태구분명                  18
좌표정보x(epsg5174)     32448
좌표정보y(epsg5174)     32448
dtype: int64

In [15]:
# 소재지 주소가 있으면 우선 사용, 없으면 도로명 주소 사용
df['주소'] = df['소재지전체주소'].fillna(df['도로명전체주소'])


In [16]:
df['주소'].isna().sum()


0

In [17]:
# 주소 문자열에서 '○○구' 패턴 추출
df['구'] = df['주소'].str.extract(r'(서울특별시\s*)?(\w+구)')[1]
# 구 정보 없는 행 제거
df[['주소', '구']].head()

,주소,구
0,서울특별시 강북구 수유동 229-46,강북구
1,서울특별시 종로구 관훈동 155-2,종로구
2,서울특별시 종로구 관훈동 155-2,종로구
3,서울특별시 종로구 인사동 241,종로구
4,서울특별시 양천구 신월동 60-33,양천구


In [ ]:
# '구' 결측치 제거
df['구'].isna().sum()
df = df[df['구'].notna()].copy()

In [ ]:
# 서울특별시 데이터만 유지 (타 지역 자동 제거)
df = df[df['주소'].str.contains('서울', na=False)].copy()


In [18]:
# 문자열 → datetime 변환
df['인허가일자'] = pd.to_datetime(df['인허가일자'], errors='coerce')
df['폐업일자'] = pd.to_datetime(df['폐업일자'], errors='coerce')

# 인허가일자 없는 행 제거
df = df[df['인허가일자'].notna()].copy()

# ✅ 2015년 이후만 사용
df = df[df['인허가일자'] >= '2015-01-01'].copy()

# 좌표 결측치 제거
df = df[
    df['좌표정보x(epsg5174)'].notna() &
    df['좌표정보y(epsg5174)'].notna()
].copy()

In [19]:
# 창업월, 창업연도
df['창업월'] = df['인허가일자'].dt.month
df['창업연도'] = df['인허가일자'].dt.year


In [20]:
# 영업 기간 계산 
df['영업일수'] = (
    df['폐업일자'].fillna(pd.Timestamp.today())
    - df['인허가일자']
).dt.days

In [21]:
#3년 이내 폐업 여부
df['폐업_3년이내'] = (
    (df['폐업일자'].notna()) &
    (df['영업일수'] <= 365 * 3)
).astype(int)

In [22]:
#문자열의 앞뒤 공백을 제거
df['업태'] = df['업태구분명'].str.strip()


In [23]:
df['업태'].value_counts()


업태
한식                 51991
기타                 38297
경양식                10920
호프/통닭               9681
일식                  8683
분식                  7272
중국식                 5180
외국음식전문점(인도,태국등)     3387
식육(숯불구이)            1392
정종/대포집/소주방          1380
김밥(도시락)              629
횟집                   615
뷔페식                  509
감성주점                 493
까페                   440
패밀리레스트랑              369
라이브카페                297
냉면집                  296
통닭(치킨)               196
출장조리                 122
키즈카페                  91
패스트푸드                 63
탕류(보신용)               61
복어취급                  26
기타 휴게음식점               2
전통찻집                   1
식품소분업                  1
Name: count, dtype: int64

In [24]:
# 업태 빈도
type_counts = df['업태'].value_counts()

# 10개 이하 업태 목록
rare_types = type_counts[type_counts <= 10].index

rare_types

Index(['기타 휴게음식점', '전통찻집', '식품소분업'], dtype='object', name='업태')

In [25]:
# 표본 10개 이하 업태 → 기타로 통합
df['업태_정리'] = df['업태'].replace(rare_types, '기타')

In [26]:
# ===============================
#  업태 그룹화 함수
# ===============================
def map_category(x):
    if pd.isna(x):
        return '기타'

    if x in ['한식', '탕류(보신용)', '냉면집', '식육(숯불구이)']:
        return '한식'

    if x in ['분식', '김밥(도시락)', '패스트푸드']:
        return '분식/간편식'

    if x in ['경양식', '패밀리레스트랑', '외국음식전문점(인도,태국등)', '뷔페식']:
        return '양식/외식'

    if x in ['중국식', '일식']:
        return '중·일식'

    if x in ['호프/통닭', '정종/대포집/소주방', '통닭(치킨)', '감성주점', '간이주점', '룸살롱']:
        return '주점/치킨'

    if x in ['까페', '커피숍', '전통찻집', '라이브카페', '다방', '키즈카페']:
        return '카페'

    return '기타'

In [27]:
# 업태 그룹 컬럼 생성
df['업태_그룹'] = df['업태'].apply(map_category)
df['업태_그룹'].value_counts()

업태_그룹
한식        53740
기타        39065
양식/외식     15185
중·일식      13863
주점/치킨     11750
분식/간편식     7964
카페          829
Name: count, dtype: int64

In [28]:
# 업태 그룹별 3년 이내 폐업률 (%)로 출력
(
    df.groupby('업태_그룹')['폐업_3년이내']
      .mean()
      .mul(100)
      .round(1)
      .sort_values()
)


업태_그룹
중·일식      27.5
카페        30.4
양식/외식     30.5
주점/치킨     31.2
한식        31.6
기타        37.3
분식/간편식    37.3
Name: 폐업_3년이내, dtype: float64

In [29]:
# 구별 폐업률
gu_rate = (
    df.groupby('구')['폐업_3년이내']
    .mean()
    .sort_values(ascending=False)
)

gu_rate.mul(100).round(1).head(10)


구
일산서구    100.0
영통구     100.0
양천구      38.6
관악구      38.2
강남구      38.1
강북구      36.8
구로구      35.8
강동구      35.3
노원구      35.1
은평구      34.5
Name: 폐업_3년이내, dtype: float64

In [30]:
# ===============================
#  구 × 업태 그룹 폐업률 Pivot
# ===============================
pivot_gu_type = df.pivot_table(
    values='폐업_3년이내',
    index='구',
    columns='업태_그룹',
    aggfunc='mean'
)


In [31]:
pivot_gu_type_pct = pivot_gu_type.mul(100).round(1)
pivot_gu_type_pct

업태_그룹,기타,분식/간편식,양식/외식,주점/치킨,중·일식,카페,한식
구,,,,,,,
강남구,45.0,42.5,34.9,34.7,29.6,34.1,36.7
강동구,37.9,41.2,31.0,33.9,27.8,45.0,35.6
강북구,42.8,38.6,23.9,34.9,31.1,21.4,32.5
강서구,30.6,34.8,35.3,30.7,32.1,15.4,35.1
관악구,34.7,39.6,39.9,40.1,35.1,35.1,38.6
광진구,31.9,41.7,35.4,31.5,28.9,32.0,32.6
구로구,43.7,36.5,30.5,29.4,29.1,33.3,37.1
금천구,28.1,37.8,30.6,26.3,28.0,20.0,31.2
노원구,44.0,35.0,32.5,34.7,25.8,27.6,32.6


In [32]:
import sys
print(sys.executable)

C:\Users\Admin\anaconda3\envs\ml-dl-nlp\python.exe


In [ ]:

# ===============================
#  히트맵 시각화
# ===============================
plt.figure(figsize=(14,10))
sns.heatmap(
    pivot_gu_type_pct,
    cmap='Reds',
    annot=False,
    cbar_kws={'label': '3년 이내 폐업률(%)'}
)
plt.title('서울 구 × 업태 그룹 3년 이내 폐업률')
plt.xlabel('업태 그룹')
plt.ylabel('구')
plt.show()


In [ ]:
# ===============================
#  위험 조합 TOP 추출
# ===============================
risk_table = pivot_gu_type_pct.stack().reset_index()
risk_table.columns = ['구', '업태_그룹', '폐업률']
risk_table.sort_values('폐업률', ascending=False).head(10)

In [ ]:
# ===============================
#  지도 시각화용 라이브러리 아나콘다에 설치.
#  아나콘다 프롬프트에서
#  conda install -c conda-forge geopandas 
# ===============================

import geopandas as gpd
gpd.__version__


In [ ]:
gu_rate_map = (
    df.groupby('구')['폐업_3년이내']
      .mean()
      .mul(100)
      .round(1)
      .reset_index()
)

gu_rate_map.sort_values('폐업_3년이내', ascending=False).head()


In [ ]:
seoul_map = gpd.read_file(
    "https://raw.githubusercontent.com/southkorea/seoul-maps/master/kostat/2013/json/seoul_municipalities_geo_simple.json"
)

seoul_map.head()


In [1]:
# 지도 쪽 구 이름 컬럼 통일
seoul_map['구'] = seoul_map['name']

# merge
map_df = seoul_map.merge(
    gu_rate_map,
    on='구',
    how='left'
)

map_df

NameError: name 'seoul_map' is not defined

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(11, 11))

# 1️⃣ 지도 색칠
map_df.plot(
    column='폐업_3년이내',
    cmap='Reds',
    linewidth=1,
    ax=ax,
    edgecolor='white',
    legend=True,
    legend_kwds={
        'label': '3년 이내 폐업률 (%)',
        'shrink': 0.6
    }
)

# 2️⃣ 구 이름 표시 (centroid 사용)
for idx, row in map_df.iterrows():
    if row['geometry'] is not None:
        x = row['geometry'].centroid.x
        y = row['geometry'].centroid.y
        ax.text(
            x, y,
            row['구'],
            fontsize=9,
            ha='center',
            va='center',
            color='black'
        )

# 3️⃣ 제목
ax.set_title(
    '서울 구별 전체 일반음식점 기준\n3년 이내 폐업률',
    fontsize=18,
    pad=12
)

ax.axis('off')
plt.show()


In [ ]:
df['생존_3년이상'] = (
    (df['폐업일자'].isna()) &
    (df['영업일수'] >= 365 * 3)
)
df['생존_3년이상'].value_counts()

In [ ]:
df['상태'] = '기타'

df.loc[df['폐업_3년이내'] == 1, '상태'] = '폐업(3년 이내)'
df.loc[df['생존_3년이상'] == True, '상태'] = '3년 이상 영업중'


In [ ]:
gdf_points = gpd.GeoDataFrame(
    df[df['상태'].isin(['폐업(3년 이내)', '3년 이상 영업중'])],
    geometry=[
        Point(xy) for xy in zip(
            df.loc[df['상태'].isin(['폐업(3년 이내)', '3년 이상 영업중']), '좌표정보x(epsg5174)'],
            df.loc[df['상태'].isin(['폐업(3년 이내)', '3년 이상 영업중']), '좌표정보y(epsg5174)']
        )
    ],
    crs="EPSG:5174"
)


In [ ]:
from shapely.geometry import Point

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[
        Point(xy) for xy in zip(
            df['좌표정보x(epsg5174)'],
            df['좌표정보y(epsg5174)']
        )
    ],
    crs="EPSG:5174"
)

gdf_points.head()


In [ ]:
# 서울 지도 좌표계 변환
seoul_map_5174 = seoul_map.to_crs(epsg=5174)

fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# 1️⃣ 배경 지도
seoul_map_5174.plot(
    ax=ax,
    color='white',
    edgecolor='lightgray',
    linewidth=1
)

# 2️⃣ 기타 영업중 (배경)
gdf_points[gdf_points['상태'] == '기타'].plot(
    ax=ax,
    markersize=2,
    color='lightgray',
    alpha=0.2,
    label='기타 영업중'
)

# 3️⃣ 🟡 3년 이상 영업중 (생존)
gdf_points[gdf_points['상태'] == '3년 이상 영업중'].plot(
    ax=ax,
    markersize=5,
    color='gold',
    alpha=0.8,
    label='3년 이상 영업중'
)

# 4️⃣ 🔴 폐업 (3년 이내)
gdf_points[gdf_points['상태'] == '폐업(3년 이내)'].plot(
    ax=ax,
    markersize=7,
    color='crimson',
    alpha=0.75,
    label='폐업(3년 이내)'
)

# 5️⃣ 구 이름 표시 (centroid)
for idx, row in seoul_map_5174.iterrows():
    x = row.geometry.centroid.x
    y = row.geometry.centroid.y
    ax.text(
        x, y,
        row['구'],
        fontsize=11,              # 글자 키움
        fontweight='bold',        # 굵게
        ha='center',
        va='center',
        color='black',
        bbox=dict(                # ⭐ 핵심
            boxstyle='round,pad=0.25',
            facecolor='white',
            edgecolor='none',
            alpha=0.85
        )
    )

# 6️⃣ 제목
ax.set_title(
    '서울 일반음식점 생존·폐업 분포 산점도\n(3년 기준)',
    fontsize=18,
    pad=12
)

ax.legend()
ax.axis('off')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot([1,2,3],[1,4,9])
plt.show()
